# Reliability analysis: paired tests and subgroup scores

This notebook compares fixed Assignment 1 prediction files. It resamples source pairs,
uses an exact paired correctness test, and measures performance within three subgroup
families. It trains no model and uses no GPU.

In [1]:
TRACK = 1
RUN_DATE = "2026-11-15"
OVERWRITE = True
SEED = 13
N_RESAMPLES = 10_000

In [2]:
from datetime import date

import pandas as pd

from dico_nli.analysis import read_prediction_file
from dico_nli.data import group_pairs, load_reference
from dico_nli.experiments import EXPERIMENTS_ROOT, experiment_dir, write_config
from dico_nli.stats import (
    bootstrap_difference,
    mcnemar_exact,
    score_subgroups,
    tag_subgroups,
)

ModuleNotFoundError: No module named 'dico_nli'

In [ ]:
train = load_reference(TRACK, "train")
dev = load_reference(TRACK, "dev")
gold = {instance.instance_id: instance.label for instance in dev}

ml_predictions = EXPERIMENTS_ROOT / "2026-09-29-ml-baseline-track1" / "predictions"
encoder_predictions = EXPERIMENTS_ROOT / "2026-09-29-deberta-base-track1" / "predictions"
systems = {
    "cues+logreg": read_prediction_file(ml_predictions / "cues+logreg.csv"),
    "both+logreg": read_prediction_file(ml_predictions / "both+logreg.csv"),
    "deberta-seed13": read_prediction_file(encoder_predictions / "seed13.csv"),
    "deberta-seed42": read_prediction_file(encoder_predictions / "seed42.csv"),
    "deberta-seed1234": read_prediction_file(encoder_predictions / "seed1234.csv"),
}

run_dir = experiment_dir(
    "reliability-track1", day=date.fromisoformat(RUN_DATE), overwrite=OVERWRITE
)
tables_dir = run_dir / "tables"
tables_dir.mkdir(exist_ok=True)

print(f"{len(dev)} dev instances; {len(group_pairs(dev))} source pairs")
print(f"Writing {run_dir}")

## Paired bootstrap intervals

Every resample draws source-pair groups with replacement, keeping reversed instances
together. The reported interval is the 2.5th to 97.5th percentile of 10,000 paired
resamples. Differences are system A minus system B.

In [ ]:
comparisons = [
    ("deberta-seed13", "both+logreg"),
    ("deberta-seed13", "deberta-seed1234"),
]
metric_names = ["weighted_f1", "macro_f1", "soft_cons", "hard_cons"]

bootstrap_rows = []
for system_a, system_b in comparisons:
    for metric in metric_names:
        difference, ci_low, ci_high = bootstrap_difference(
            dev,
            systems[system_a],
            systems[system_b],
            metric,
            n_resamples=N_RESAMPLES,
            seed=SEED,
        )
        bootstrap_rows.append(
            {
                "system_a": system_a,
                "system_b": system_b,
                "metric": metric,
                "difference": difference,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "excludes_zero": ci_low > 0 or ci_high < 0,
            }
        )

bootstrap = pd.DataFrame(bootstrap_rows)
bootstrap.to_csv(tables_dir / "bootstrap_intervals.csv", index=False)
bootstrap.round(4)

## Exact McNemar tests

The two systems predict the same dev instances, so the test uses only discordant
correctness outcomes. The p-value is the exact two-sided binomial probability.

In [ ]:
mcnemar_rows = []
for system_a, system_b in comparisons:
    a_wrong_b_right, a_right_b_wrong, p_value = mcnemar_exact(
        gold, systems[system_a], systems[system_b]
    )
    mcnemar_rows.append(
        {
            "system_a": system_a,
            "system_b": system_b,
            "a_wrong_b_right": a_wrong_b_right,
            "a_right_b_wrong": a_right_b_wrong,
            "p_value": p_value,
        }
    )

mcnemar = pd.DataFrame(mcnemar_rows)
mcnemar.to_csv(tables_dir / "mcnemar_results.csv", index=False)
mcnemar

## Subgroup scores

Phrase overlap and total length are properties of the unordered source pair, so both
directions receive the same tag. Length buckets use the nearest observed total-word
counts to the 33rd and 67th percentiles of unique dev source pairs. The length-cue tag
comes from `dico_nli.analysis.analyze_predictions`. We report classification metrics
only within subgroups; filtering by the cue can split the unit used by SoftCons and
HardCons.

In [ ]:
tags = tag_subgroups(dev, train)
boundaries = tags.attrs["length_bucket_boundaries"]
tags.to_csv(tables_dir / "subgroup_tags.csv", index=False)

subgroup_tables = []
for column in ["cue_status", "phrase_overlap", "length_bucket"]:
    table = score_subgroups(dev, systems, tags, column)
    table.insert(0, "subgroup_type", column)
    subgroup_tables.append(table)

subgroup_scores = pd.concat(subgroup_tables, ignore_index=True)
subgroup_scores.to_csv(tables_dir / "subgroup_scores.csv", index=False)

print(f"Length buckets: short <= {boundaries[0]}, medium <= {boundaries[1]}, long > {boundaries[1]}")
subgroup_scores.round(4)

In [ ]:
write_config(
    run_dir,
    seed=SEED,
    track=TRACK,
    command=(
        "PYTHONIOENCODING=utf-8 uv run jupyter execute "
        "notebooks/assignment-2/02_reliability_track1.ipynb"
    ),
    extra={
        "n_resamples": N_RESAMPLES,
        "bootstrap_unit": "source pair",
        "confidence_interval": "95% percentile",
        "systems": list(systems),
        "comparisons": [list(comparison) for comparison in comparisons],
        "length_bucket_boundaries_total_words": list(boundaries),
    },
)
print("Reliability run complete.")